[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/stokes_swimmer/stokes_swimmer.ipynb)

# Swimming without Inertia

This example computes how a swimmer moves through a viscous fluid without inertia, Purcell's three-link swimmer beside a reciprocal flapper. In slow, thick flow, such as a bacterium in water or a person in honey, nothing coasts, so a swimmer that stops changing its shape stops at once, and at every instant the forces its body exerts on the fluid add up to no net force and no net torque. The swimmer is modelled as a chain of beads, each pushing on the fluid through a regularized Stokeslet, the flow of a force spread over a blob of the bead's size. Because the body's motion is linear in the rate at which its shape changes, a stroke that goes back along the path it came undoes itself, which is Purcell's scallop theorem, while a swimmer that takes its two hinges around a loop advances with every cycle.

The beads are points and the body's motion is a rate, a bivector, so `sites.commutator(Twist)` is the map from the body's rate to every bead's velocity. Its adjugate sums the beads' forces into the forque on the body, because a forque paired with a rate is the power the forces deliver to beads moving at that rate, `rigid_velocities.adjugate()(forces) & rate`, with no lever arms and no reference point. One solve with the resistance `rigid_velocities.adjugate()(bead_mobility.solve(rigid_velocities))` gives the body's rate that keeps the forque zero, and the poses are the running product of the step motors `(body_rate * (-dt / 2)).exp()`.

In the matrix notation of Stokesian dynamics, as in [Cortez, 2001](https://www.math.tulane.edu/~cortez/Prints/stokeslet.html) and [Lauga, 2011](https://arxiv.org/abs/1011.3051), the bead velocities $\mathbf U = \mathsf M\,\mathbf F$ come from a $2N\times 2N$ mobility matrix of blocks $\mathsf M_{ij} \propto \big((r_{ij}^2 + 2\epsilon^2)\,\mathsf I + \mathbf r_{ij}\mathbf r_{ij}^{\mathsf T}\big)/(r_{ij}^2 + \epsilon^2)^{3/2}$, rigid motion enters through the $2N\times 3$ matrix $\mathsf K$ with rows $\big(\mathsf I \;\; \mathsf J\,(\mathbf x_i - \mathbf x_0)\big)$, $\mathsf J = \begin{pmatrix}0 & -1\\ 1 & 0\end{pmatrix}$, about a chosen reference point $\mathbf x_0$, and the force and torque constraints $\mathsf K^{\mathsf T}\mathbf F = \big(\sum_i \mathbf F_i,\ \sum_i (\mathbf x_i - \mathbf x_0)\times\mathbf F_i\big) = 0$ give the body's velocity and angular velocity as $(\mathbf V, \Omega) = -(\mathsf K^{\mathsf T}\mathsf M^{-1}\mathsf K)^{-1}\,\mathsf K^{\mathsf T}\mathsf M^{-1}\,\mathbf u^{\mathrm{shape}}$. Written as $(\mathbf V, \Omega) = \mathsf A(\boldsymbol\theta)\,\dot{\boldsymbol\theta}$ over the hinge angles, this is the Shapere–Wilczek connection, and the net motion per cycle is the path-ordered exponential of $\oint \mathsf A\,d\boldsymbol\theta$ in $\mathrm{SE}(2)$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, concatenate
from numga.algebras import PGA2D as ga
from examples.util.animation import save_figure, show_animation
from examples.fluids.stokes_swimmer import render

mv = NumpyContext(ga).multivector
Point = ga.gatype.antivector()                             # Antivector-
Twist = ga.gatype.bivector()                               # Bivector+
# A force is a plane through the origin, extrinsic, read by its pairing with a velocity, an ideal point.
Force = ga.gatype.vector().derive.nondegenerate            # Vector+
Direction = Force.derive.dual                              # Antivector-
Mobility = ga.gatype((Direction, Force))                   # Direction- <- Force+
origin = mv.w.dual()                                       # [] Point-

## The mobility map

A push on the fluid at one bead sets all of the fluid moving, less and less with distance, and the regularized Stokeslet spreads each bead's force over a blob of the bead's size so that the flow stays finite at the bead itself. Over every pair of beads these flows make one map from the forces the beads push with to the velocities they move at. The body's own motion gives every bead a velocity, `sites.commutator(Twist)`, and the adjugate of that map sums the beads' forces into one forque on the body, because a forque paired with a rate is the power the forces deliver to beads moving at that rate. As the swimmer changes its shape, its body moves so that this forque stays zero, which is one solve for the body's rate.

In [ ]:
# The swimmer: the half-length of its centre link, the beads' offsets along the centre link and along each arm,
# and the radius of the blob each bead's force is spread over.
hinge_half_span = 0.8
centre_offsets = np.array([-0.5, 0.0, 0.5])
arm_offsets = np.array([0.35, 0.7, 1.05])
bead_radius = 0.14

In [ ]:
def stokeslet(targets: Point, sources: Point, radius: float) -> Mobility:
    """The fluid velocity at the targets per unit force spread over a blob of the given radius at the sources."""
    # The difference of two points of one weight is a direction; as one, the map from forces to velocities is
    # square, as its solves need.
    separations = (targets - sources).cast(Direction)                       # [...] Direction-
    # The squared distance, the squared norm of the line joining the two points, softened by the radius: the
    # blob spreads each force over the bead's size, which keeps the flow finite where the force acts.
    blob_squares = (targets & sources).norm_squared() + radius**2           # [...] Scalar+
    # The fluid moves along the force, and further along the separation by the force's pairing with it;
    # both parts fall off with distance.
    return ((blob_squares + radius**2) * Force.dual() + separations * (Force & separations)) / (blob_squares * blob_squares.square_root())


def mobility(sites: Point, radius: float) -> Mobility:
    """The map from all the beads' forces to all their velocities, its blocks the Stokeslets between every pair of
    beads, formed over the beads as a batch and assembled into a map between the two fields."""
    beads = sites.batch()                                                   # [..., n_beads] Point-
    return stokeslet(beads[..., :, None], beads[..., None, :], radius).field(0, 1)   # [...] Direction-[n_beads] <- Force+[n_beads]

In [ ]:
def force_free_motion(sites: Point, shape_velocities: Direction, radius: float) -> tuple[Twist, Force]:
    """The body's rate in its own frame and the beads' forces of a swimmer free of force and torque, as its beads
    change shape at the given velocities."""
    bead_mobility = mobility(sites, radius)                                 # [...] Direction-[n_beads] <- Force+[n_beads]
    # Over `Twist`, the space of all rates, each bead's commutator with the rate is that bead's velocity.
    rigid_velocities = sites.commutator(Twist)                              # [...] Direction-[n_beads] <- Twist+
    # The forces that carry the beads along with the body at a given rate, summed into the forque on the
    # body: the adjugate carries forces back across the map that carried the rate out to velocities.
    resistance = rigid_velocities.adjugate()(bead_mobility.solve(rigid_velocities))   # [...] Forque- <- Twist+
    # Changing shape alone would push on the fluid too; the body moves at the rate whose forque cancels it.
    body_rate = -resistance.solve(rigid_velocities.adjugate()(bead_mobility.solve(shape_velocities)))   # [...] Twist+
    # The forces the beads push with under both motions together, balanced by construction.
    forces = bead_mobility.solve(rigid_velocities(body_rate) + shape_velocities)   # [...] Force+[n_beads]
    return body_rate, forces

## Purcell's stroke beside a reciprocal stroke

Two swimmers stroke for three cycles. Purcell's swimmer swings its two hinges a quarter cycle apart, a loop in the space of hinge angles, while the reciprocal flapper swings them together, back and forth along a line. With nothing coasting, the body's rate is linear in the rate at which the shape changes, so how far a stroke carries the swimmer depends only on the path its shape takes, and going back along a path undoes it.

In [ ]:
# The stroke's amplitude in radians, the cycles, the steps per cycle; the points the flow is drawn at; the steps
# per frame, and the duration of a frame.
amplitude = 1.05
n_cycles = 3
n_steps_per_cycle = 60
n_grid_columns, n_grid_rows = 81, 55
grid_x = np.linspace(-2.8, 2.8, n_grid_columns)
grid_y = np.linspace(-1.9, 1.9, n_grid_rows)
n_steps_per_frame = 2
duration_ms = 40

In [ ]:
def three_link_shape(
    hinge_half_span: float,
    centre_offsets: np.ndarray,
    arm_offsets: np.ndarray,
    angles: np.ndarray,
    rates: np.ndarray,
) -> tuple[Point, Direction]:
    """The beads of a three-link swimmer at the given hinge angles, and their velocities in the body's frame as the
    hinges turn at the given angular rates."""
    # The beads lie along the swimmer's axis, through the origin along x, at their offsets from the origin.
    axis = mv.direction([1, 0])                                                 # [] Direction-
    centre_beads = origin + axis * centre_offsets                               # [3] Point-
    # A hinge is the meet of two planes through it, the plane along the swimmer's axis and the plane across it, a
    # bivector generating turns about itself.
    left_hinge = (mv.x + hinge_half_span * mv.w) ^ mv.y                         # [] Twist+
    right_hinge = (mv.x - hinge_half_span * mv.w) ^ mv.y                        # [] Twist+
    left_turn = (left_hinge * (angles[..., 0, None] / 2)).exp()                 # [..., 1] Motor+
    right_turn = (right_hinge * (-angles[..., 1, None] / 2)).exp()              # [..., 1] Motor+

    left_rest = origin - axis * (hinge_half_span + arm_offsets[::-1])           # [3] Point-
    right_rest = origin + axis * (hinge_half_span + arm_offsets)                # [3] Point-
    left_beads = left_turn >> left_rest                                         # [..., 3] Point-
    right_beads = right_turn >> right_rest                                      # [..., 3] Point-
    centre_sites = centre_beads.broadcast_to(angles.shape[:-1] + centre_beads.shape)   # [..., 3] Point-

    # Each arm turns about its hinge at the hinge's rate, so a bead's velocity is its commutator with that
    # rate; the left hinge's angle counts the other way round. The centre link does not move in the body's
    # frame.
    left_velocities = left_beads.commutator(-left_hinge * rates[..., 0, None])  # [..., 3] Direction-
    right_velocities = right_beads.commutator(right_hinge * rates[..., 1, None])   # [..., 3] Direction-
    centre_velocities = 0 * centre_sites.commutator(right_hinge)                # [..., 3] Direction-

    sites = concatenate([left_beads, centre_sites, right_beads], axis=-1)      # [..., 9] Point-
    velocities = concatenate([left_velocities, centre_velocities, right_velocities], axis=-1)   # [..., 9] Direction-
    return sites.field(), velocities.field()


n_steps = n_cycles * n_steps_per_cycle
dt = 2 * np.pi / n_steps_per_cycle
phases = np.arange(n_steps) * dt                                                # [n_steps]

# Purcell's swimmer takes its hinges around a circle; the flapper swings them together along a line.
angles = np.stack([
    np.stack([amplitude * np.cos(phases), amplitude * np.sin(phases)], axis=-1),
    np.stack([amplitude * np.cos(phases), -0.5 * amplitude * np.cos(phases)], axis=-1),
], axis=1)                                                                      # [n_steps, 2, 2]
rates = np.stack([
    np.stack([-amplitude * np.sin(phases), amplitude * np.cos(phases)], axis=-1),
    np.stack([-amplitude * np.sin(phases), 0.5 * amplitude * np.sin(phases)], axis=-1),
], axis=1)                                                                      # [n_steps, 2, 2]

sites, shape_velocities = three_link_shape(hinge_half_span, centre_offsets, arm_offsets, angles, rates)
body_rate, forces = force_free_motion(sites, shape_velocities, bead_radius)     # [n_steps, 2] Twist+, Force+[n_beads]

In [ ]:
def flow_field(sample_points: Point, bead_points: Point, bead_forces: Force, radius: float) -> Direction:
    """The fluid velocity at the sample points: the map from the beads' forces to the flow at each sample point,
    applied to the forces, sums the flows of all the beads."""
    return stokeslet(sample_points[..., None], bead_points.batch(), radius).field(1)(bead_forces)   # [...] Direction-


# Each step moves the body in its own frame, so it multiplies the pose on the right. A running product
# puts later factors on the left; taking it over the steps' reverses and reversing the result puts them
# on the right.
start_motor = (mv.xy * np.zeros(2)).exp()                                       # [2] Motor+
step_motors = (body_rate * (-dt / 2)).exp()                                     # [n_steps, 2] Motor+
later_poses = start_motor * step_motors.reverse().cumprod(axis=0).reverse()     # [n_steps, 2] Motor+
motors = concatenate([start_motor[None], later_poses], axis=0)                  # [n_steps + 1, 2] Motor+
positions = motors >> origin                                                    # [n_steps + 1, 2] Point-
# The beads and their forces carried from the body's frame into the world; a force is moved as its
# complement, a direction, and complemented again.
world_sites = motors[:-1] >> sites                                              # [n_steps, 2] Point-[n_beads]
world_forces = (motors[:-1] >> forces.dual()).dual()                            # [n_steps, 2] Force+[n_beads]

grid = mv.point(np.stack(np.meshgrid(grid_x, grid_y, indexing="xy"), axis=-1))  # [n_grid_rows, n_grid_columns] Point-
first_flow = flow_field(grid, world_sites[0, :, None, None], world_forces[0, :, None, None], bead_radius)   # [2, n_grid_rows, n_grid_columns] Direction-
save_figure(render.draw_summary(angles, positions, world_sites, grid, first_flow), "stokes_swimmer");

The left panel shows the instantaneous flow around Purcell's swimmer at the start of its stroke, with the trail its centre leaves over the three cycles. The middle panel follows both centres along x, Purcell's swimmer (solid) stepping steadily ahead while the reciprocal flapper (dashed) lunges forward and back and ends every cycle where it started. The right panel draws the two strokes in hinge-angle space, Purcell's circle and the flapper's line.

In [ ]:
def flows(grid: Point) -> Iterator[Direction]:
    """Per frame, both swimmers' flows at the points of the grid."""
    for step in range(0, n_steps, n_steps_per_frame):
        yield flow_field(grid, world_sites[step, :, None, None], world_forces[step, :, None, None], bead_radius)


show_animation(render.animate_swimmers(positions, world_sites, grid, flows(grid), n_steps_per_frame), "stokes_swimmer", duration_ms)

Both panels show the instantaneous fluid velocity as streaks that fade downstream, their length proportional to speed on one fixed scale, with Purcell's swimmer above and the reciprocal flapper below. The flapper's flow stops whenever its stroke turns around, and the fixed seeds then show as specks. The streaks are a texture of the instantaneous field, drawn afresh at every frame, and follow no fluid particles.

In [ ]:
# checks
# Every step is free of force and torque.
rigid_velocities = sites.commutator(Twist)                                      # [n_steps, 2] Direction-[n_beads] <- Twist+
np.testing.assert_allclose(rigid_velocities.adjugate()(forces).kernel, 0.0, atol=1e-9)
# The reciprocal flapper returns to its starting pose after every cycle; Purcell's swimmer advances by the same
# nonzero distance each cycle.
cycle_motors = motors[::n_steps_per_cycle]                                      # [n_cycles + 1, 2] Motor+
np.testing.assert_allclose((cycle_motors[1:, 1] - cycle_motors[0, 1]).kernel, 0.0, atol=1e-4)
cycle_centres = positions[::n_steps_per_cycle, 0]                               # [n_cycles + 1] Point-
cycle_advances = mv.x & (cycle_centres[1:] - cycle_centres[:-1])                # [n_cycles] Scalar+
assert (cycle_advances.abs() > 0.04).all()
assert ((cycle_advances - cycle_advances[0]).abs() < 1e-3).all()